In [1]:
import importlib
import tiktoken
print(f"tiktoken version : {importlib.metadata.version("tiktoken")}")

tiktoken version : 0.8.0


In [2]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)
# 주어진 text를 tokenizer를 통해 token을 얻는다.
tokenizer = tiktoken.get_encoding("gpt2")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [4]:
strings = tokenizer.decode(integers)

In [13]:
with open("datas/the-verdict.txt","r",encoding="utf-8") as f:
    raw_text = f.read()

context_size = 4
enc_text = tokenizer.encode(raw_text)
enc_sample = enc_text[50:]
x = enc_sample[0:context_size]
y = enc_sample[1:1+context_size]
print(f"x: {x}")
print(f"y:   {y}")

x: [290, 4920, 2241, 287]
y:   [4920, 2241, 287, 257]


In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
# data set을 만든다 input ids랑 target ids
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "토큰의 길이는 max_length 보다 커야합니다."
        for i in range(0, len(token_ids) - max_length, stride):
            # self-supervised 데이터는 바로뒤에꺼
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]    


In [18]:
import tiktoken
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text=txt, tokenizer=tokenizer, max_length=max_length, stride=stride)
    dataloader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers) 
    return dataloader   


In [19]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False, drop_last=False, num_workers=0)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
secont_batch = next(data_iter)
print(secont_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [26]:
import torch
import torch.nn

input_ids = torch.tensor([2,3,5,1])
vocab_size = 6
output_dim = 3

embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
print(f"embedding_layer weights : {embedding_layer.weight}")
token_embeddings = embedding_layer(input_ids)
print(token_embeddings)


embedding_layer weights : Parameter containing:
tensor([[-1.5169, -0.7342, -2.0010],
        [ 0.6491,  0.0535, -0.2131],
        [ 0.1491,  1.4684, -2.0814],
        [ 0.1944,  0.4985,  0.4189],
        [-1.4103, -0.2155,  1.0901],
        [ 2.7814, -0.5249,  1.7636]], requires_grad=True)
tensor([[ 0.1491,  1.4684, -2.0814],
        [ 0.1944,  0.4985,  0.4189],
        [ 2.7814, -0.5249,  1.7636],
        [ 0.6491,  0.0535, -0.2131]], grad_fn=<EmbeddingBackward0>)


In [28]:
vocab_size = 50257
output_dim = 256
batch_size = 8
max_length = 4
stride = 4
shuffle = False
drop_last = True
num_workers = 0

dataloader = create_dataloader_v1(raw_text, batch_size=batch_size, max_length=max_length, stride=stride, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

data_iter = iter(dataloader)
first_batch = next(data_iter)
input_ids = first_batch[0]

token_embedding_layer = torch.nn.Embedding(vocab_size,output_dim)
token_embeddings = token_embedding_layer(input_ids)
pos_embedding_layer = torch.nn.Embedding(max_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(4))
print(f"token_embeddings : {token_embeddings.shape}")
print(f"pos_embeddings : {pos_embeddings.shape}")

input_embeddings = token_embeddings + pos_embeddings
print(f"input_embeddings : {input_embeddings.shape}")

token_embeddings : torch.Size([8, 4, 256])
pos_embeddings : torch.Size([4, 256])
input_embeddings : torch.Size([8, 4, 256])
